# Training an LSTM-based Authorized Push Payment (APP) fraud detection model 
This notebook is used to prepare and preprocess the datasets from IBM Synthetic Datasets, which are then used for training an LSTM-based Authorized Push Payment (APP) fraud detection model. The datasets are loaded from four different CSV files and then combined into one DataFrame. This enriched dataset provides the features needed to train a fraud detection model that can learn patterns from account characteristics, bank metrics, and transaction behaviors. Such as:
- Transaction details (amount, currency, format, type)
- Fraud labels (Is_APP_Fraud)
- Sender account features (country, currency, entity type, overdraft, branch/bank metrics)
- Recipient account features (country, entity type)

In [ ]:
import os
import pandas as pd

datasets_dir = "datasets/"
#processed_datasets_dir = "datasets_processed/"
#os.makedirs(processed_datasets_dir, exist_ok=True)

## Data preparation

### Load datasets into Pandas DataFrame

In [ ]:
# ============================================================================
# Define columns to load for each dataset
# ============================================================================

banks_cols = [
    "Bank_ID",     "Num_Transactions",     "Num_Total_Locations"
]

accts_cols = [
    "Financial_Institution_ID",     "Financial_Institution_Name",     "Branch", 
    "Account_ID",     "Account_Country",     "Account_Currency",     "Entity_Type", 
    "Account_Type",     "Max_Overdraft"
]

bank_xfers_cols = [
    "Transaction_Number",
    "Transaction_Date",    "Transaction_Time",    "Transaction_Day_of_Week",    "From_Bank",
    "From_Account",     "To_Bank",    "To_Account",    "Amount_Paid",     "Payment_Currency",
    "Payment_Format",     "Transaction_Type",     "Is_Instant_Payments_Fraud",
    "Is_APP_Fraud",      "Sufficient_Funds",     "Overdraft_Okay",     "Is_Hold",     "From_End_Balance"
]
# ============================================================================
# Load datasets with only necessary columns
# ============================================================================

banks_df = pd.read_csv(datasets_dir + "sd299_us_small_banks.csv", usecols=banks_cols)
accts_people_df = pd.read_csv(datasets_dir + "sd299_us_small_liquid_accts_people.csv", usecols=accts_cols)
accts_companies_df = pd.read_csv(datasets_dir + "sd299_us_small_liquid_accts_companies.csv", encoding='iso8859_2', usecols=accts_cols)
#accts_companies_df = pd.read_csv(datasets_dir + "sd299_us_small_liquid_accts_companies.csv", usecols=accts_cols)
bank_xfers_df = pd.read_csv(datasets_dir + "sd299_us_small_bank_xfers-chrono.csv", usecols=bank_xfers_cols)


### Prepare ***banks*** dataset

In [ ]:
banks_df.info()

### Prepare ***liquid_accts*** from ***liquid_accts_people*** and ***liquid_accts_companies*** datasets

In [ ]:
accts_people_df.info()

In [ ]:
accts_companies_df.info()

In [ ]:
# Combine liquid_accts_people and liquid_accts_companies into one DataFrame
accts_df = pd.concat([accts_people_df, accts_companies_df], ignore_index=True)
accts_df.info()


In [ ]:
# Convert columns to intended data types
accts_df = accts_df.astype({
    "Account_Country": "category", 
    "Account_Currency": "category", 
    "Entity_Type": "category", 
    "Account_Type": "category", 
    "Account_ID": str
})

accts_df.info()


In [ ]:
# Filter out rows of "Cash" Account_Type
#accts_df = accts_df[accts_df["Account_Type"] != "Cash"]


### Prepare ***branches*** dataset from ***liquid_accts*** and ***banks***

In [ ]:
# Derive branches dataset from liquid_accts
branches_df = accts_df.groupby(
    ["Financial_Institution_ID", "Branch"])["Account_ID"].nunique().rename("Branch_Account_Count").reset_index()

# Merge required banks columns into branches dataset
branches_df = branches_df.merge(
    banks_df[["Bank_ID", "Num_Transactions", "Num_Total_Locations"]], 
    left_on="Financial_Institution_ID", right_on="Bank_ID", how="left")
branches_df.drop("Bank_ID", axis="columns", inplace=True)

# Rename columns to identify the source of data
branches_df.rename(
    columns={
        "Num_Transactions": "Bank_Num_Transactions", 
        "Num_Total_Locations": "Bank_Num_Total_Locations"
    }, inplace=True)

# Convert columns to intended data types
branches_df = branches_df.astype({
    "Bank_Num_Transactions": "Int64",
    "Bank_Num_Total_Locations": "Int64"
})

#branches_df.to_csv(processed_datasets_dir + "branches.csv", index=False)

branches_df.info()


### Merge columns from ***branches*** into ***liquid_accts*** dataset

In [ ]:
accts_merged_df = accts_df.merge(branches_df, on=["Financial_Institution_ID", "Branch"], how="left")

#accts_merged_df.to_csv(processed_datasets_dir + "liquid_accts_merged.csv", index=False)

accts_merged_df.info()


### Prepare ***bank_xfers*** dataset

In [ ]:
bank_xfers_df.info()

In [ ]:
# Remove rows of Cash transactions
bank_xfers_cleaned_df = bank_xfers_df[(bank_xfers_df["From_Bank"] != "Cash") & (bank_xfers_df["To_Bank"] != "Cash")]

# Convert columns to intended data types
bank_xfers_cleaned_df = bank_xfers_cleaned_df.astype({
    "From_Account": str,
    "To_Account": str,
    "Payment_Currency": "category",
    "Payment_Format": "category",
    "Transaction_Type": "category",
    "Is_APP_Fraud": bool,
    "Sufficient_Funds": bool,
    "Overdraft_Okay": bool,
    "Is_Hold": bool
})

bank_xfers_cleaned_df.reset_index(drop=True, inplace=True)
bank_xfers_cleaned_df.info()

### Merge columns from ***liquid_accts*** into ***bank_xfers*** dataset

In [ ]:
# Merge in intended liquid_accts columns for From_Account
bank_xfers_merged_df = bank_xfers_cleaned_df.merge(
    accts_merged_df[["Financial_Institution_Name", "Account_ID", 
                     "Account_Country", "Account_Currency", "Entity_Type", "Account_Type", "Max_Overdraft",
                     "Branch_Account_Count", "Bank_Num_Transactions", "Bank_Num_Total_Locations"]], 
                     left_on=["From_Bank", "From_Account"], right_on=["Financial_Institution_Name", "Account_ID"], how="left")
bank_xfers_merged_df.drop(["Financial_Institution_Name", "Account_ID"], axis="columns", inplace=True)

# Rename merged columns to identify the source of data
bank_xfers_merged_df.rename(
    columns={"Account_Country": "From_Account_Country", 
             "Account_Currency": "From_Account_Currency", 
             "Entity_Type": "From_Entity_Type",
             "Account_Type": "From_Account_Type",
             "Max_Overdraft": "From_Account_Max_Overdraft", 
             "Branch_Account_Count": "From_Branch_Account_Count",
             "Bank_Num_Transactions": "From_Bank_Num_Transactions",
             "Bank_Num_Total_Locations": "From_Bank_Num_Total_Locations"}, inplace=True) 

# Combine Date and Time into a Timestamp
bank_xfers_merged_df["Timestamp"] = (bank_xfers_merged_df["Transaction_Date"].astype(str) + " " +
    bank_xfers_merged_df["Transaction_Time"].astype(str))

# Merge in intended liquid_accts columns for To_Account
bank_xfers_merged_df = bank_xfers_merged_df.merge(
    accts_merged_df[["Financial_Institution_Name", "Account_ID", 
                     "Account_Country", "Entity_Type"]], 
                     left_on=["To_Bank", "To_Account"], right_on=["Financial_Institution_Name", "Account_ID"], how="left")
bank_xfers_merged_df.drop(["Financial_Institution_Name", "Account_ID"], axis="columns", inplace=True)

# Rename merged columns to identify the source of data
bank_xfers_merged_df.rename(
    columns={"Account_Country": "To_Account_Country", 
             "Entity_Type": "To_Entity_Type"}, inplace=True) 

#bank_xfers_merged_df = bank_xfers_merged_df[(bank_xfers_merged_df["From_Bank"] != "Dummy Securities Transac Counterparty") & 
#                                            (bank_xfers_merged_df["To_Bank"] != "Dummy Securities Transac Counterparty")]

bank_xfers_merged_df.reset_index(drop=True, inplace=True)
bank_xfers_merged_df.info()

In [ ]:
#bank_xfers_merged_df.to_csv(processed_datasets_dir + "bank_xfers_merged.csv", index=False)

## Data preprocessing

In [ ]:
import pandas as pd
import numpy as np

#datasets_dir = "datasets_processed/"

### Load dataset into Pandas DataFrame

In [ ]:
#dataset_df = pd.read_csv(datasets_dir + "bank_xfers_merged.csv")
#dataset_df.info()
dataset_df = bank_xfers_merged_df


### Convert data in DataFrames before passing to Pre-processing Pipeline

In [ ]:
def log_transform(x):
    """Apply sign-preserving log transformation to monetary amounts."""
    return np.sign(x) * np.log1p(np.abs(x))


In [ ]:
dataset_df["Timestamp"] = pd.to_datetime(dataset_df["Timestamp"], format="%Y-%m-%d %H:%M:%S.%f")
# Extract temporal components
dataset_df["Month"] = dataset_df["Timestamp"].dt.month
dataset_df["Day_Of_Month"] = dataset_df["Timestamp"].dt.day
dataset_df["Day_Of_Week"] = dataset_df["Timestamp"].dt.dayofweek
dataset_df["Hour"] = dataset_df["Timestamp"].dt.hour
dataset_df["Minute"] = dataset_df["Timestamp"].dt.minute
dataset_df["Is_Weekday"] = dataset_df["Timestamp"].dt.dayofweek.isin([0, 1, 2, 3, 4]).astype(int)

# Cyclical encoding for temporal features (pre-computed)
# Month (12-month cycle)
dataset_df["Month_Sin"] = np.sin(2 * np.pi * dataset_df["Month"] / 12)
dataset_df["Month_Cos"] = np.cos(2 * np.pi * dataset_df["Month"] / 12)

# Day of month (31-day cycle)
dataset_df["Day_Sin"] = np.sin(2 * np.pi * dataset_df["Day_Of_Month"] / 31)
dataset_df["Day_Cos"] = np.cos(2 * np.pi * dataset_df["Day_Of_Month"] / 31)

# Day of week (7-day cycle)
dataset_df["DayOfWeek_Sin"] = np.sin(2 * np.pi * dataset_df["Day_Of_Week"] / 7)
dataset_df["DayOfWeek_Cos"] = np.cos(2 * np.pi * dataset_df["Day_Of_Week"] / 7)

# Time of day (24-hour cycle)
seconds = dataset_df["Hour"] * 3600 + dataset_df["Minute"] * 60
dataset_df["TimeOfDay_Sin"] = np.sin(2 * np.pi * seconds / 86400)
dataset_df["TimeOfDay_Cos"] = np.cos(2 * np.pi * seconds / 86400)

dataset_df["Amount_Paid_Log"] = log_transform(dataset_df["Amount_Paid"])
dataset_df["From_End_Balance_Log"] = log_transform(dataset_df["From_End_Balance"])
dataset_df["From_Account_Max_Overdraft_Log"] = log_transform(
    dataset_df["From_Account_Max_Overdraft"].fillna(0)
)

### Create preprocessor using ColumnTransformer for all columns used in the training  

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# Define feature groups
temporal_cyclical_features = [
    "Month_Sin", "Month_Cos",  "Day_Sin", "Day_Cos",
    "DayOfWeek_Sin", "DayOfWeek_Cos", "TimeOfDay_Sin", "TimeOfDay_Cos"
]

boolean_features = ["Is_Weekday", "Sufficient_Funds", "Overdraft_Okay", "Is_Hold"]

amount_features = [
    "Amount_Paid_Log",   "From_End_Balance_Log",  "From_Account_Max_Overdraft_Log"
]

count_features = [
    "From_Branch_Account_Count",   "From_Bank_Num_Transactions",   "From_Bank_Num_Total_Locations"
]

# Low-cardinality categorical features (use OneHotEncoder)
low_card_categorical = [
    "From_Entity_Type",  "From_Account_Type",   "To_Entity_Type",   "Payment_Format",   "Transaction_Type"
]

# High-cardinality categorical features (use OrdinalEncoder as PMML-compatible alternative)
high_card_categorical = [
    "From_Account_Country",    "From_Account_Currency",  "To_Account_Country",  "Payment_Currency"
]

# Create the PMML-compatible preprocessor
preprocessor = ColumnTransformer([
    ("temporal_cyclical", MinMaxScaler(), temporal_cyclical_features),
    ("boolean", "passthrough", boolean_features),
    ("amounts", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value=0)),
        ("scale", MinMaxScaler())
    ]), amount_features),
    ("counts", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", MinMaxScaler())
    ]), count_features),
    ("low_card_cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), low_card_categorical),
    ("high_card_cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ("scale", MinMaxScaler())
    ]), high_card_categorical)
], remainder="drop")
#    ["Is_APP_Fraud"]

preprocessor.fit(dataset_df)

### Pickle the preprocessor

In [ ]:
import joblib
import os

save_dir = "saved_preprocessor"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(preprocessor, open(os.path.join(save_dir, "fitted_preprocessor.pkl"), "wb"))
preprocessor = joblib.load(open(os.path.join(save_dir, "fitted_preprocessor.pkl"), "rb"))

In [ ]:
preprocessed_sample = preprocessor.transform(dataset_df[:100])
preprocessed_size = preprocessed_sample.shape[-1]
print(preprocessed_size)

###  Export the preprocessor to PMML format

In [ ]:
from sklearn2pmml import sklearn2pmml
from sklearn2pmml.pipeline import PMMLPipeline

# Create a PMML pipeline
pmml_pipeline = PMMLPipeline([
    ("preprocessor", preprocessor)
])

# Export to PMML
sklearn2pmml(pmml_pipeline, "preprocessor.pmml", with_repr=True)

## LSTM Model Training Section

## Set up TensorFlow environment

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import keras
import tensorflow as tf

print(f"\nTensorFlow version: {tf.__version__}")

# Configuration for LSTM training
timesteps = 7  # Number of past transactions to consider
model_save_dir = 'saved_model'
os.makedirs(model_save_dir, exist_ok=True)

## Prepare data for LSTM training

In [ ]:
# Extract labels (Is_APP_Fraud) before preprocessing
labels = dataset_df['Is_APP_Fraud'].values

# Get preprocessed features
X_preprocessed = preprocessor.transform(dataset_df)

# Ensure the data is in the correct dtype (float32) for TensorFlow
X_preprocessed = X_preprocessed.astype(np.float32)

input_size = X_preprocessed.shape[1]

print(f"Preprocessed feature size: {input_size}")
print(f"Total samples: {len(labels)}")

# Split data into train, validation, and test sets (50%, 30%, 20%)
total_samples = len(labels)
train_size = int(total_samples * 0.5)
val_size = int(total_samples * 0.3)
test_size = total_samples - train_size - val_size

print(f"Train size: {train_size}, Validation size: {val_size}, Test size: {test_size}")

# IMPORTANT: Data is grouped by account and chronological within each account
# The generators will handle sequence creation properly
np.random.seed(1111)
indices = np.arange(total_samples)

# Split indices while preserving order (account grouping)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

print("\nWARNING: Current implementation assumes data remains in account-grouped order.")
print("Sequences are created from consecutive indices, which works correctly")
print("only if transactions are grouped by account in the dataset.")

## Utility functions and classes for LSTM

In [ ]:
def print_trainable_parameters(model):
    """Print trainable parameters in the model."""
    total = 0
    for variable in model.trainable_variables():
        shape = variable.get_shape()
        parameters = 1
        for dim in shape:
            parameters *= dim
        total += parameters
        print(variable, shape, parameters)
    print(f"Total trainable parameters: {total}")


def f1(conf):
    """Calculate F1 score from confusion matrix."""
    precision = float(conf[1][1]) / (conf[1][1] + conf[0][1])
    recall = float(conf[1][1]) / (conf[1][1] + conf[1][0])
    return 2 * precision * recall / (precision + recall)


class TP(keras.metrics.TruePositives):
    """Custom True Positives metric for last timestep."""
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[-1, :, :], y_pred[-1, :, :], sample_weight)


class FP(keras.metrics.FalsePositives):
    """Custom False Positives metric for last timestep."""
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[-1, :, :], y_pred[-1, :, :], sample_weight)


class FN(keras.metrics.FalseNegatives):
    """Custom False Negatives metric for last timestep."""
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[-1, :, :], y_pred[-1, :, :], sample_weight)


class TN(keras.metrics.TrueNegatives):
    """Custom True Negatives metric for last timestep."""
    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(y_true[-1, :, :], y_pred[-1, :, :], sample_weight)

## Generator for training batches
    Generate training batches for the LSTM model.
    
    Generator function produces one batch of data at a time using indices, instead
    of loading everything into memory at once. This allows streaming of large
    datasets for on-the-fly processing.

In [ ]:
def gen_training_batch(X_data, y_data, indices, batch_size, timesteps):
    fraud_mask = y_data[indices] == 1
    fraud_indices = indices[fraud_mask]
    non_fraud_indices = indices[~fraud_mask]

    fsize = len(fraud_indices)

    while True:
        # Balance the dataset by sampling equal fraud and non-fraud
        if fsize > 0 and len(non_fraud_indices) >= fsize:
            selected_indices = np.concatenate((
                fraud_indices,
                np.random.choice(non_fraud_indices, fsize, replace=False)
            ))
        else:
            selected_indices = indices.copy()
        
        # Create sequences - filter for valid indices with enough history
        valid_indices = []
        for idx in selected_indices:
            if idx >= timesteps - 1:  # Ensure we have enough history
                valid_indices.append(idx)
        
        valid_indices = np.array(valid_indices)
        rows = len(valid_indices)
        
        if rows == 0:
            continue
        
        # Create index array for sequences
        # Each row represents one sequence of 7 consecutive transactions
        index_array = np.zeros((rows, timesteps), dtype=int)
        for i in range(timesteps):
            index_array[:, i] = valid_indices - (timesteps - 1 - i)
        
        # Get data and targets
        data_buffer = X_data[index_array.flatten()].reshape(rows, timesteps, -1)
        target_buffer = y_data[index_array.flatten()].reshape(rows, timesteps, 1)
        
        # Shuffle the sequences (not the raw indices) to randomize training order
        shuffle_idx = np.arange(rows)
        np.random.shuffle(shuffle_idx)
        data_buffer = data_buffer[shuffle_idx]
        target_buffer = target_buffer[shuffle_idx]
        
        # Yield batches
        batch_ptr = 0
        while (batch_ptr + batch_size) <= rows:
            data = data_buffer[batch_ptr:batch_ptr + batch_size]
            targets = target_buffer[batch_ptr:batch_ptr + batch_size]
            batch_ptr += batch_size
            yield data, targets

## Generator for validation/test batches
Generate test batches for model evaluation.

In [ ]:
def gen_test_batch(X_data, y_data, indices, batch_size, timesteps):
    # Filter indices to ensure we have enough history
    valid_indices = indices[indices >= timesteps - 1]
    rows = len(valid_indices)
    
    if rows == 0:
        return
    
    # Create index array for sequences
    index_array = np.zeros((rows, timesteps), dtype=int)
    for i in range(timesteps):
        index_array[:, i] = valid_indices - (timesteps - 1 - i)
    
    count = 0
    while (count + batch_size <= rows):
        batch_indices = index_array[count:count + batch_size].flatten()
        data = X_data[batch_indices].reshape(batch_size, timesteps, -1)
        targets = y_data[batch_indices].reshape(batch_size, timesteps, 1)
        count += batch_size
        yield data, targets


## Define and compile the LSTM model

In [ ]:
# LSTM (Long Short-Term Memory) model parameters
# input_shape=tf_input - is a matrix of timesteps (the number of time steps per
# input sequence), and input_size is the number of features per time step.
# Training and Testing is completed in batches.

units = [200, 200]
output_size = 1
batch_size = 16
tf_input = (timesteps, input_size)

lstm_model = keras.models.Sequential([
    keras.layers.LSTM(units[0], return_sequences=True, input_shape=tf_input),
    keras.layers.LSTM(units[1], return_sequences=True),
    keras.layers.Dense(output_size, activation='sigmoid')
])

print("\nModel summary:")
lstm_model.summary()

# Compile model with metrics
metrics = [
    'accuracy',
    TP(name='TP'),
    FP(name='FP'),
    FN(name='FN'),
    TN(name='TN'),
    keras.metrics.TruePositives(name='tp'),
    keras.metrics.FalsePositives(name='fp'),
    keras.metrics.FalseNegatives(name='fn'),
    keras.metrics.TrueNegatives(name='tn')
]

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=metrics)

## Train the LSTM model

In [ ]:
print("\nTraining LSTM model...")

steps_per_epoch = 500
checkpoint_dir = "./checkpoints/app_fraud_lstm/"
filepath = checkpoint_dir + "iter-{epoch:02d}.weights.h5"

print("Starting training...")
cp_callback = keras.callbacks.ModelCheckpoint(
    filepath=filepath,
    save_weights_only=True,
    verbose=1
)

train_generate = gen_training_batch(X_preprocessed, labels, train_indices, batch_size, timesteps)

lstm_model.fit(
    train_generate,
    epochs=5,
    steps_per_epoch=steps_per_epoch,
    verbose=1,
    callbacks=[cp_callback]
)

print(f"\nSaving model weights to {model_save_dir}/wts.weights.h5...")
lstm_model.save_weights(os.path.join(model_save_dir, "wts.weights.h5"))
print(f"Saving complete model to {model_save_dir}/model.keras...")
lstm_model.save(os.path.join(model_save_dir, "model.keras"))

## Test the LSTM model

In [ ]:
# Create a new model with larger batch size for testing
test_batch_size = 2000

test_model = keras.models.Sequential([
    keras.layers.LSTM(units[0], return_sequences=True, input_shape=tf_input),
    keras.layers.LSTM(units[1], return_sequences=True),
    keras.layers.Dense(output_size, activation='sigmoid')
])

print(f"Loading model weights from {model_save_dir}/wts.weights.h5...")
test_model.load_weights(os.path.join(model_save_dir, "wts.weights.h5"))
test_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=metrics)

print("\nValidation set evaluation:")
# Calculate number of valid sequences and steps for validation set
val_valid_indices = val_indices[val_indices >= timesteps - 1]
val_steps = len(val_valid_indices) // test_batch_size
print(f"Validation samples: {len(val_valid_indices)}, Steps: {val_steps}")

val_generate = gen_test_batch(X_preprocessed, labels, val_indices, test_batch_size, timesteps)
val_results = test_model.evaluate(val_generate, steps=val_steps, verbose=1)

print("\nTest set evaluation:")
# Calculate number of valid sequences and steps for test set
test_valid_indices = test_indices[test_indices >= timesteps - 1]
test_steps = len(test_valid_indices) // test_batch_size
print(f"Test samples: {len(test_valid_indices)}, Steps: {test_steps}")

test_generate = gen_test_batch(X_preprocessed, labels, test_indices, test_batch_size, timesteps)
test_results = test_model.evaluate(test_generate, steps=test_steps, verbose=1)

## Calculate and display detailed metrics

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

print("\n" + "="*80)
print("DETAILED METRICS")
print("="*80)

# Generate predictions for validation set
print("\nGenerating predictions for validation set...")
val_generate = gen_test_batch(X_preprocessed, labels, val_indices, test_batch_size, timesteps)
val_predictions = []
val_true_labels = []

for batch_data, batch_labels in val_generate:
    preds = test_model.predict(batch_data, verbose=0)
    # Take the last timestep predictions
    val_predictions.extend(preds[:, -1, 0])
    val_true_labels.extend(batch_labels[:, -1, 0])

val_predictions = np.array(val_predictions)
val_true_labels = np.array(val_true_labels)
val_pred_classes = (val_predictions > 0.5).astype(int)

print("\n--- VALIDATION SET METRICS ---")
print(f"Total samples: {len(val_true_labels)}")
print(f"Positive samples: {np.sum(val_true_labels)}")
print(f"Negative samples: {len(val_true_labels) - np.sum(val_true_labels)}")

# Confusion Matrix
val_cm = confusion_matrix(val_true_labels, val_pred_classes)
print("\nConfusion Matrix:")
print(f"                 Predicted Negative  Predicted Positive")
print(f"Actual Negative       {val_cm[0][0]:6d}              {val_cm[0][1]:6d}")
print(f"Actual Positive       {val_cm[1][0]:6d}              {val_cm[1][1]:6d}")

# Calculate metrics
val_precision = precision_score(val_true_labels, val_pred_classes, zero_division=0)
val_recall = recall_score(val_true_labels, val_pred_classes, zero_division=0)
val_f1 = f1_score(val_true_labels, val_pred_classes, zero_division=0)

print(f"\nPrecision: {val_precision:.4f}")
print(f"Recall:    {val_recall:.4f}")
print(f"F1-Score:  {val_f1:.4f}")

print("\nClassification Report:")
print(classification_report(val_true_labels, val_pred_classes,
                          target_names=['Non-Fraud', 'Fraud'],
                          zero_division=0))

# Generate predictions for test set
print("\n--- TEST SET METRICS ---")
print("Generating predictions for test set...")
test_generate = gen_test_batch(X_preprocessed, labels, test_indices, test_batch_size, timesteps)
test_predictions = []
test_true_labels = []

for batch_data, batch_labels in test_generate:
    preds = test_model.predict(batch_data, verbose=0)
    # Take the last timestep predictions
    test_predictions.extend(preds[:, -1, 0])
    test_true_labels.extend(batch_labels[:, -1, 0])

test_predictions = np.array(test_predictions)
test_true_labels = np.array(test_true_labels)
test_pred_classes = (test_predictions > 0.5).astype(int)

print(f"Total samples: {len(test_true_labels)}")
print(f"Positive samples: {np.sum(test_true_labels)}")
print(f"Negative samples: {len(test_true_labels) - np.sum(test_true_labels)}")

# Confusion Matrix
test_cm = confusion_matrix(test_true_labels, test_pred_classes)
print("\nConfusion Matrix:")
print(f"                 Predicted Negative  Predicted Positive")
print(f"Actual Negative       {test_cm[0][0]:6d}              {test_cm[0][1]:6d}")
print(f"Actual Positive       {test_cm[1][0]:6d}              {test_cm[1][1]:6d}")

# Calculate metrics
test_precision = precision_score(test_true_labels, test_pred_classes, zero_division=0)
test_recall = recall_score(test_true_labels, test_pred_classes, zero_division=0)
test_f1 = f1_score(test_true_labels, test_pred_classes, zero_division=0)

print(f"\nPrecision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")

print("\nClassification Report:")
print(classification_report(test_true_labels, test_pred_classes,
                          target_names=['Non-Fraud', 'Fraud'],
                          zero_division=0))

print("\n" + "="*80)


## Convert to ONNX format
tf2onnx is a tool that converts TensorFlow models into ONNX (Open Neural Network
Exchange) format, making them compatible with other platforms and backends. ONNX
is an open-source format designed for interoperability across different deep
learning frameworks, allowing models trained in TensorFlow to be used on different
platforms without needing to be re-trained or re-implemented.

In [ ]:
try:
    import tf2onnx
    import onnx
    
    # For Keras 3.x compatibility, convert from saved model format
    # First, ensure we have a saved model
    saved_model_path = os.path.join(model_save_dir, "model.keras")
    
    # Create a temporary SavedModel format (required for tf2onnx)
    temp_saved_model_dir = os.path.join(model_save_dir, "temp_saved_model")
    test_model.export(temp_saved_model_dir)
    
    # Convert from SavedModel to ONNX
    onnx_output_path = os.path.join(model_save_dir, "app_fraud_lstm.onnx")
    
    # Use command-line style conversion which is more reliable
    import subprocess
    cmd = [
        "python3", "-m", "tf2onnx.convert",
        "--saved-model", temp_saved_model_dir,
        "--output", onnx_output_path,
        "--opset", "13"
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"Model successfully converted and saved to {onnx_output_path}")
    else:
        print(f"ONNX conversion failed: {result.stderr}")
        print("Skipping ONNX conversion...")
    
    # Clean up temporary directory
    import shutil
    if os.path.exists(temp_saved_model_dir):
        shutil.rmtree(temp_saved_model_dir)
        
except Exception as e:
    print(f"ONNX conversion error: {e}")
    print("Skipping ONNX conversion. Model is still saved in Keras format.")

print("\n" + "="*80)
print("Training and testing complete!")
print("="*80)
print(f"\nSaved files:")
print(f"  - Preprocessor: {os.path.join(save_dir, 'fitted_preprocessor.pkl')}")
print(f"  - PMML: preprocessor.pmml")
print(f"  - Model weights: {os.path.join(model_save_dir, 'wts.weights.h5')}")
print(f"  - Complete model: {os.path.join(model_save_dir, 'model.keras')}")
onnx_path = os.path.join(model_save_dir, "app_fraud_lstm.onnx")
if os.path.exists(onnx_path):
    print(f"  - ONNX model: {onnx_path}")